# A Tiny Spatial Gate for DarkIR: run everything in Colab

Run the cells from top to bottom. **Runtime > Change runtime type > T4 GPU** first.

Rough times on a free GPU: setup 5 min, baseline eval 15 min, one training run 20 min, one full test eval 15 min.
If you only want to check that it works, run the cells marked *quick check* and skip the long ones.

## 0. Settings (edit these two lines)

In [ ]:
REPO_URL = "https://github.com/YOUR-USERNAME/darkir-spatial-gate"   # <- your GitHub repo link
LOLBLUR  = "/content/drive/MyDrive/DarkIR_project/LOLBlur Dataset"     # <- folder in Drive that holds test.zip and train_separate_zip/
SAVE     = "/content/drive/MyDrive/DarkIR_gate_outputs"                # checkpoints and results are saved here (so they survive a restart)

## 1. Setup: Drive, DarkIR code, our files, packages, pretrained weights

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs(SAVE + '/checkpoints', exist_ok=True)
os.makedirs(SAVE + '/results', exist_ok=True)

%cd /content
!git clone -q https://github.com/cidautai/DarkIR.git
!git clone -q {REPO_URL} gate_repo
%cd /content/DarkIR

# our files go on top of the original repo (nothing of the original is changed)
!cp /content/gate_repo/archs/DarkIR_gate.py archs/
!cp /content/gate_repo/train_gate.py /content/gate_repo/eval_gate.py /content/gate_repo/significance.py /content/gate_repo/make_subset.py .

# the original requirements pin old versions, so we loosen them
!sed -i -E 's/==([0-9]+\.[0-9]+(\.[0-9]+)?)/>=\1/' requirements.txt
!pip install -q -r requirements.txt
!pip install -q lpips ptflops pytorch-msssim huggingface_hub
# only needed if the repo's test utils complain about the nccl backend
!sed -i 's/"nccl"/"gloo"/' utils/test_utils.py

from huggingface_hub import snapshot_download
snapshot_download(repo_id='Cidaut/DarkIR', local_dir='models')
!ls -la models/DarkIR_384.pt

## 2. Data: test set and our training subset

In [ ]:
import os
os.makedirs('/content/data', exist_ok=True)
!unzip -q -o "{LOLBLUR}/test.zip" -d /content/data
!ls /content/data/test
!python make_subset.py "{LOLBLUR}/train_separate_zip" /content/data/train_sub
!ls /content/data/train_sub

## 3. Original DarkIR baseline

*Quick check:* the first cell scores only 50 images. The second cell does all 1800 (about 15 min).

In [ ]:
!python eval_gate.py --name pretrained --mode pretrained --ckpt models/DarkIR_384.pt --limit 50 --out {SAVE}/results

In [ ]:
!python eval_gate.py --name pretrained_full --mode pretrained --ckpt models/DarkIR_384.pt --out {SAVE}/results
# expected (paper): PSNR 27.0029, SSIM 0.8829, LPIPS 0.1620

## 4. Train the gate with L1 loss (about 20 min per seed)

In [ ]:
!python train_gate.py --mode gate1 --tag _amp2 --freeze_body --seed 0 --out {SAVE}/checkpoints

In [ ]:
# second seed (optional)
!python train_gate.py --mode gate1 --tag _amp2 --freeze_body --seed 1 --out {SAVE}/checkpoints

## 5. Train the gate with the combined loss (L1 + 0.2 SSIM term + 0.2 LPIPS)

In [ ]:
!python train_gate.py --mode gate1 --tag _combo --loss combo --freeze_body --seed 0 --out {SAVE}/checkpoints

## 6. Score the trained gates on the full test set (about 15 min each)

In [ ]:
C = SAVE + '/checkpoints'
!python eval_gate.py --name gate1_amp2_full         --mode gate1 --ckpt {C}/gate1_amp2_s0_best.pt       --out {SAVE}/results
!python eval_gate.py --name gate1_amp2_s1_full      --mode gate1 --ckpt {C}/gate1_amp2_s1_best.pt       --out {SAVE}/results
!python eval_gate.py --name gate1_combo_s0_last_full --mode gate1 --ckpt {C}/gate1_combo_s0_last.pt     --out {SAVE}/results

## 7. Is the difference bigger than noise? (per-scene comparison)

In [ ]:
import os
os.environ['RESULTS_DIR'] = SAVE + '/results'
!python significance.py pretrained_full gate1_amp2_full gate1_amp2_s1_full gate1_combo_s0_last_full

## Notes
- Numbers can differ slightly from the paper because of GPU randomness.
- The 30-epoch run and the first-round (whole network fine-tuned) experiment are described in the README.